# BTK Datathon 2026 — 02 Model Comparison

This notebook is the second step after `01_data_understanding_baseline.ipynb`.

Main goal: compare stronger models using a **clean validation setup**. The focus is not only getting a lower local CV MSE, but also avoiding validation leakage that can make Kaggle performance worse than expected.

Reference result from Notebook 01:

- Mean baseline MSE: ~230.61
- Ridge + TF-IDF CV MSE: ~87.77
- Ridge + TF-IDF CV RMSE: ~9.37

Target for this notebook: try to beat ~87.77 MSE with controlled feature engineering and model comparison.

In [9]:
import os
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge, ElasticNet
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor

RANDOM_STATE = 42
N_SPLITS = 5

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

## 1. Load data

The notebook expects competition files to be placed under `../data/`:

```text
data/train.csv
data/test_x.csv
data/sample_submission.csv
```

If you run it from a different working directory, update `DATA_DIR`.

In [10]:
DATA_DIR = Path("../data")
TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test_x.csv"
SAMPLE_SUB_PATH = DATA_DIR / "sample_submission.csv"

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUB_PATH)

TARGET = "career_success_score"
ID_COL = "student_id"
TEXT_COL = "mentor_feedback_text"

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Sample submission shape:", sample_submission.shape)

display(train.head())

Train shape: (10000, 47)
Test shape: (10000, 46)
Sample submission shape: (2, 2)


,student_id,application_year,age,graduation_year,department,university_tier,cgpa,english_exam_score,attendance_rate,failed_courses_count,target_role,coding_score,problem_solving_score,data_structures_score,sql_score,machine_learning_score,backend_score,frontend_score,cloud_score,devops_score,project_quality_score,real_client_project_count,internship_count,internship_duration_months,freelance_project_count,hackathon_count,hackathon_awards,portfolio_score,github_repo_count,github_avg_stars,open_source_contribution_count,linkedin_profile_score,cv_quality_score,technical_interview_score,hr_interview_score,communication_score,teamwork_score,leadership_score,presentation_score,certification_count,bootcamp_count,applications_sent,interviews_attended,hobby,preferred_social_media_platform,career_success_score,mentor_feedback_text
0,STU_000001,2021,21,2021,Computer Engineering,Tier 4,3.17,62.54,77.31,0,DevOps Engineer,73.28,71.11,52.91,84.980000,81.77,62.710000,71.570000,63.041897,69.952625,81.90,0,3,11.0,0,0,0,65.54,18,1.85,10.0,86.58,42.06,40.57,50.29,79.83,44.14,62.70,58.84,3,1,24,0,photography,LinkedIn,86.78,Proje kalitesi ve makine öğrenimi konusundaki ...
1,STU_000002,2024,20,2024,Computer Engineering,Tier 4,3.24,75.10,87.13,3,Backend Developer,63.12,78.90,61.81,37.450740,65.54,69.944694,60.830000,64.510000,57.940000,24.68,0,0,NaN,1,1,0,54.48,7,1.22,1.0,33.34,65.39,82.99,67.43,43.60,22.05,42.32,40.54,2,0,46,5,reading,YouTube,46.16,Kodlama ve problem çözme becerileri gelişmekte...
2,STU_000003,2024,28,2024,Electrical Electronics Engineering,Tier 4,3.00,68.53,95.64,1,Frontend Developer,100.00,86.44,83.62,85.440000,87.18,80.580000,96.433149,62.220000,81.750000,78.92,2,0,0.0,2,0,0,75.10,4,12.12,2.0,61.37,52.25,43.06,20.19,48.62,65.64,47.27,82.56,1,2,46,5,cinema,Reddit,84.08,İleri düzey frontend geliştirme becerileri ile...
3,STU_000004,2019,22,2018,Computer Engineering,Tier 1,2.82,54.85,77.80,2,Backend Developer,99.08,72.15,77.15,89.214871,69.49,85.751415,72.860000,73.680000,54.080000,54.93,0,1,9.0,0,1,0,82.40,4,2.96,3.0,45.15,24.12,32.06,28.00,59.84,3.89,78.69,85.05,2,4,49,7,running,Reddit,89.97,Güçlü bir kodlama yeteneği ve backend geliştir...
4,STU_000005,2026,22,2026,Computer Engineering,Tier 3,2.28,72.25,71.97,1,Product Analyst,92.65,91.15,84.51,70.700000,74.11,80.620000,86.830000,80.340000,87.560000,72.85,2,0,NaN,2,0,0,48.02,14,0.97,12.0,74.86,74.83,71.82,65.14,63.30,52.86,27.22,84.29,1,0,119,13,football,X,92.46,Ürün analizi alanına olan tutkusu ve makine öğ...


## 2. Why Kaggle score can be worse than local CV

Local CV is an estimate, not the real leaderboard score. It can look too optimistic if:

- preprocessing is fitted on full train before CV,
- TF-IDF/SVD/encoding is fitted outside folds,
- test data is used while designing transformations,
- too many submissions are used to tune to the public leaderboard,
- the public leaderboard split is not representative of the private split.

So in this notebook, transformations are kept inside sklearn pipelines where possible.

In [11]:
print(train[TARGET].describe())

mean_pred = np.full(len(train), train[TARGET].mean())
mean_mse = mean_squared_error(train[TARGET], mean_pred)
print(f"Mean baseline MSE: {mean_mse:.4f}")
print(f"Mean baseline RMSE: {np.sqrt(mean_mse):.4f}")

count    10000.000000
mean        76.942507
std         15.186669
min          0.000000
25%         66.937500
50%         77.810000
75%         88.472500
max        100.000000
Name: career_success_score, dtype: float64
Mean baseline MSE: 230.6119
Mean baseline RMSE: 15.1859


## 3. Row-wise feature engineering

These features are safe because they are calculated row-by-row from existing columns. They do not use the target and do not fit statistics from the full dataset.

The aim is to expose useful signals to models:

- technical skill averages and spread,
- social skill averages,
- project / internship / hackathon experience,
- portfolio and GitHub activity,
- interview composite,
- role-specific skill alignment,
- application-to-interview efficiency,
- missingness indicators for meaningful missing fields.

In [12]:
TECH_COLS = [
    "coding_score", "problem_solving_score", "data_structures_score",
    "sql_score", "machine_learning_score", "backend_score",
    "frontend_score", "cloud_score", "devops_score"
]

SOCIAL_COLS = [
    "communication_score", "teamwork_score", "leadership_score", "presentation_score"
]

MISSING_FLAG_COLS = [
    "english_exam_score", "internship_duration_months", "github_avg_stars",
    "open_source_contribution_count", "hr_interview_score",
    "linkedin_profile_score", "portfolio_score"
]

ROLE_SKILL_MAP = {
    "Data Scientist": ["machine_learning_score", "sql_score", "problem_solving_score", "coding_score"],
    "Data Analyst": ["sql_score", "problem_solving_score", "communication_score", "presentation_score"],
    "Machine Learning Engineer": ["machine_learning_score", "coding_score", "problem_solving_score", "cloud_score"],
    "Backend Developer": ["backend_score", "coding_score", "data_structures_score", "sql_score"],
    "Frontend Developer": ["frontend_score", "coding_score", "presentation_score", "communication_score"],
    "Full Stack Developer": ["backend_score", "frontend_score", "coding_score", "data_structures_score"],
    "DevOps Engineer": ["devops_score", "cloud_score", "backend_score", "problem_solving_score"],
    "Cloud Engineer": ["cloud_score", "devops_score", "backend_score", "problem_solving_score"],
    "Data Engineer": ["sql_score", "backend_score", "cloud_score", "data_structures_score"],
    "Software Engineer": ["coding_score", "problem_solving_score", "data_structures_score", "backend_score"],
    "AI Engineer": ["machine_learning_score", "coding_score", "cloud_score", "problem_solving_score"],
}

def add_rowwise_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Missingness flags: sometimes missing values carry meaning.
    for col in MISSING_FLAG_COLS:
        if col in df.columns:
            df[f"{col}_missing"] = df[col].isna().astype(int)

    # Zero-fill activity-like columns where missing can reasonably mean no recorded activity.
    zero_like_cols = ["internship_duration_months", "github_avg_stars", "open_source_contribution_count"]
    for col in zero_like_cols:
        if col in df.columns:
            df[col] = df[col].fillna(0)

    # Technical skill aggregates.
    df["tech_mean"] = df[TECH_COLS].mean(axis=1)
    df["tech_std"] = df[TECH_COLS].std(axis=1)
    df["tech_min"] = df[TECH_COLS].min(axis=1)
    df["tech_max"] = df[TECH_COLS].max(axis=1)
    df["tech_range"] = df["tech_max"] - df["tech_min"]
    df["tech_top3_mean"] = df[TECH_COLS].apply(lambda r: r.nlargest(3).mean(), axis=1)

    # Social skill aggregates.
    df["social_mean"] = df[SOCIAL_COLS].mean(axis=1)
    df["social_std"] = df[SOCIAL_COLS].std(axis=1)

    # Interview composite.
    df["interview_mean"] = df[["technical_interview_score", "hr_interview_score"]].mean(axis=1)
    df["technical_minus_hr"] = df["technical_interview_score"] - df["hr_interview_score"]

    # Experience / portfolio / academic composites. These are simple, interpretable indices.
    df["experience_index"] = (
        df["real_client_project_count"] * 2.0 +
        df["internship_count"] * 2.5 +
        df["internship_duration_months"] * 0.4 +
        df["freelance_project_count"] * 1.2 +
        df["hackathon_count"] * 1.2 +
        df["hackathon_awards"] * 2.5 +
        df["open_source_contribution_count"] * 0.4
    )

    df["portfolio_activity_index"] = (
        df["portfolio_score"].fillna(df["portfolio_score"].median()) +
        df["github_repo_count"] * 0.4 +
        df["github_avg_stars"] * 0.4 +
        df["open_source_contribution_count"] * 0.4 +
        df["linkedin_profile_score"].fillna(df["linkedin_profile_score"].median()) * 0.5 +
        df["cv_quality_score"] * 0.5
    )

    df["academic_index"] = (
        df["cgpa"] * 20 +
        df["attendance_rate"] * 0.2 +
        df["english_exam_score"].fillna(df["english_exam_score"].median()) * 0.2 -
        df["failed_courses_count"] * 3
    )

    df["application_efficiency"] = np.where(
        df["applications_sent"] > 0,
        df["interviews_attended"] / df["applications_sent"],
        0
    )

    df["learning_count"] = df["certification_count"] + df["bootcamp_count"]
    df["years_from_graduation"] = df["application_year"] - df["graduation_year"]

    # Role-specific skill alignment.
    def role_skill_mean(row):
        role = row.get("target_role")
        cols = ROLE_SKILL_MAP.get(role, TECH_COLS)
        existing_cols = [c for c in cols if c in row.index]
        return row[existing_cols].mean() if existing_cols else np.nan

    df["role_skill_mean"] = df.apply(role_skill_mean, axis=1)
    df["role_skill_gap"] = df["role_skill_mean"] - df["tech_mean"]

    # Text surface features. Actual TF-IDF is handled inside the pipeline.
    txt = df[TEXT_COL].fillna("").astype(str)
    df["mentor_text_len"] = txt.str.len()
    df["mentor_word_count"] = txt.str.split().str.len()

    return df

# Quick check
train_fe = add_rowwise_features(train.drop(columns=[TARGET]))
print("Original feature count:", train.drop(columns=[TARGET]).shape[1])
print("Feature count after row-wise engineering:", train_fe.shape[1])
display(train_fe.head(3))

Original feature count: 46
Feature count after row-wise engineering: 73


,student_id,application_year,age,graduation_year,department,university_tier,cgpa,english_exam_score,attendance_rate,failed_courses_count,target_role,coding_score,problem_solving_score,data_structures_score,sql_score,machine_learning_score,backend_score,frontend_score,cloud_score,devops_score,project_quality_score,real_client_project_count,internship_count,internship_duration_months,freelance_project_count,hackathon_count,hackathon_awards,portfolio_score,github_repo_count,github_avg_stars,open_source_contribution_count,linkedin_profile_score,cv_quality_score,technical_interview_score,hr_interview_score,communication_score,teamwork_score,leadership_score,presentation_score,certification_count,bootcamp_count,applications_sent,interviews_attended,hobby,preferred_social_media_platform,mentor_feedback_text,english_exam_score_missing,internship_duration_months_missing,github_avg_stars_missing,open_source_contribution_count_missing,hr_interview_score_missing,linkedin_profile_score_missing,portfolio_score_missing,tech_mean,tech_std,tech_min,tech_max,tech_range,tech_top3_mean,social_mean,social_std,interview_mean,technical_minus_hr,experience_index,portfolio_activity_index,academic_index,application_efficiency,learning_count,years_from_graduation,role_skill_mean,role_skill_gap,mentor_text_len,mentor_word_count
0,STU_000001,2021,21,2021,Computer Engineering,Tier 4,3.17,62.54,77.31,0,DevOps Engineer,73.28,71.11,52.91,84.98000,81.77,62.710000,71.570000,63.041897,69.952625,81.90,0,3,11.0,0,0,0,65.54,18,1.85,10.0,86.58,42.06,40.57,50.29,79.83,44.14,62.70,58.84,3,1,24,0,photography,LinkedIn,Proje kalitesi ve makine öğrenimi konusundaki ...,0,0,0,0,0,0,0,70.147169,9.815953,52.91000,84.98,32.07000,80.010000,61.3775,14.672129,45.430,-9.72,15.9,141.800,91.370,0.000000,4,0,66.703630,-3.443539,225,30
1,STU_000002,2024,20,2024,Computer Engineering,Tier 4,3.24,75.10,87.13,3,Backend Developer,63.12,78.90,61.81,37.45074,65.54,69.944694,60.830000,64.510000,57.940000,24.68,0,0,0.0,1,1,0,54.48,7,1.22,1.0,33.34,65.39,82.99,67.43,43.60,22.05,42.32,40.54,2,0,46,5,reading,YouTube,Kodlama ve problem çözme becerileri gelişmekte...,0,1,0,0,0,0,0,62.227270,11.118139,37.45074,78.90,41.44926,71.461565,37.1275,10.129684,75.210,15.56,2.8,107.533,88.246,0.108696,2,0,58.081359,-4.145912,259,33
2,STU_000003,2024,28,2024,Electrical Electronics Engineering,Tier 4,3.00,68.53,95.64,1,Frontend Developer,100.00,86.44,83.62,85.44000,87.18,80.580000,96.433149,62.220000,81.750000,78.92,2,0,0.0,2,0,0,75.10,4,12.12,2.0,61.37,52.25,43.06,20.19,48.62,65.64,47.27,82.56,1,2,46,5,cinema,Reddit,İleri düzey frontend geliştirme becerileri ile...,0,0,0,0,0,0,0,84.851461,10.685677,62.22000,100.00,37.78000,94.537716,61.0225,16.614637,31.625,22.87,7.2,139.158,89.834,0.108696,3,0,81.903287,-2.948174,232,28


## 4. Define feature groups and preprocessing

Important detail: `FunctionTransformer(add_rowwise_features)` is inside the model pipeline. This means feature engineering is applied consistently inside each fold during cross-validation.

In [13]:
def get_feature_groups(df: pd.DataFrame):
    df_fe = add_rowwise_features(df)
    feature_df = df_fe.drop(columns=[ID_COL], errors="ignore")

    numeric_cols = feature_df.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = feature_df.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

    # mentor_feedback_text is handled separately by TF-IDF, not as a normal categorical column.
    categorical_cols = [c for c in categorical_cols if c != TEXT_COL]

    return numeric_cols, categorical_cols, TEXT_COL

X = train.drop(columns=[TARGET])
y = train[TARGET].copy()
X_test = test.copy()

numeric_cols, categorical_cols, text_col = get_feature_groups(X)

print("Numeric columns:", len(numeric_cols))
print("Categorical columns:", categorical_cols)
print("Text column:", text_col)

Numeric columns: 66
Categorical columns: ['department', 'university_tier', 'target_role', 'hobby', 'preferred_social_media_platform']
Text column: mentor_feedback_text


In [14]:
def make_preprocessor(tfidf_max_features=3000):
    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    text_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="constant", fill_value="")),
        ("to_1d", FunctionTransformer(lambda x: x.ravel(), validate=False)),
        ("tfidf", TfidfVectorizer(
            max_features=tfidf_max_features,
            ngram_range=(1, 2),
            min_df=2,
            sublinear_tf=True
        ))
    ])

    return ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_cols),
            ("cat", categorical_transformer, categorical_cols),
            ("text", text_transformer, [text_col])
        ],
        remainder="drop"
    )

def make_pipeline(model, tfidf_max_features=3000):
    return Pipeline(steps=[
        ("feature_engineering", FunctionTransformer(add_rowwise_features, validate=False)),
        ("preprocess", make_preprocessor(tfidf_max_features=tfidf_max_features)),
        ("model", model)
    ])

## 5. Model comparison

We start with models that are reasonable for this data:

- Ridge: strong sparse baseline for TF-IDF + one-hot data
- ElasticNet: another linear baseline
- LightGBM / XGBoost / CatBoost: stronger nonlinear models if installed
- ExtraTrees / RandomForest: sanity-check tree ensembles

Do not blindly trust one lucky CV result. Check fold stability too.

In [15]:
models = {}

models["Ridge_TFIDF"] = make_pipeline(
    Ridge(alpha=10.0),
    tfidf_max_features=3000
)

models["ElasticNet_TFIDF"] = make_pipeline(
    ElasticNet(alpha=0.001, l1_ratio=0.05, random_state=RANDOM_STATE, max_iter=5000),
    tfidf_max_features=3000
)

# Optional libraries. If one is not installed, the notebook simply skips it.
try:
    import lightgbm as lgb
    models["LightGBM_TFIDF"] = make_pipeline(
        lgb.LGBMRegressor(
            n_estimators=1200,
            learning_rate=0.03,
            num_leaves=31,
            max_depth=-1,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.1,
            reg_lambda=1.0,
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbose=-1
        ),
        tfidf_max_features=1500
    )
    print("LightGBM available")
except Exception as e:
    print("LightGBM skipped:", e)

try:
    import xgboost as xgb
    models["XGBoost_TFIDF"] = make_pipeline(
        xgb.XGBRegressor(
            n_estimators=1000,
            learning_rate=0.03,
            max_depth=4,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.1,
            reg_lambda=1.0,
            objective="reg:squarederror",
            random_state=RANDOM_STATE,
            n_jobs=-1,
            verbosity=0
        ),
        tfidf_max_features=1200
    )
    print("XGBoost available")
except Exception as e:
    print("XGBoost skipped:", e)

try:
    from catboost import CatBoostRegressor
    models["CatBoost_TFIDF"] = make_pipeline(
        CatBoostRegressor(
            iterations=1200,
            learning_rate=0.03,
            depth=6,
            l2_leaf_reg=5,
            loss_function="RMSE",
            random_seed=RANDOM_STATE,
            verbose=0
        ),
        tfidf_max_features=1000
    )
    print("CatBoost available")
except Exception as e:
    print("CatBoost skipped:", e)

print("Models to compare:", list(models.keys()))

LightGBM available
XGBoost available
CatBoost available
Models to compare: ['Ridge_TFIDF', 'ElasticNet_TFIDF', 'LightGBM_TFIDF', 'XGBoost_TFIDF', 'CatBoost_TFIDF']


In [16]:
cv = KFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

results = []

for name, model in models.items():
    print(f"\nRunning {name}...")
    neg_mse_scores = cross_val_score(
        model,
        X,
        y,
        scoring="neg_mean_squared_error",
        cv=cv,
        n_jobs=-1,
        error_score="raise"
    )
    mse_scores = -neg_mse_scores
    row = {
        "model": name,
        "mean_mse": mse_scores.mean(),
        "rmse": np.sqrt(mse_scores.mean()),
        "std_mse": mse_scores.std(),
        "fold_scores": mse_scores
    }
    results.append(row)
    print("Fold MSE:", np.round(mse_scores, 4))
    print(f"Mean MSE: {row['mean_mse']:.4f} | RMSE: {row['rmse']:.4f} | Std: {row['std_mse']:.4f}")

results_df = pd.DataFrame(results).sort_values("mean_mse")
display(results_df[["model", "mean_mse", "rmse", "std_mse"]])


Running Ridge_TFIDF...
Fold MSE: [89.225  91.3432 83.2115 83.2469 89.7833]
Mean MSE: 87.3620 | RMSE: 9.3468 | Std: 3.4451

Running ElasticNet_TFIDF...
Fold MSE: [89.4657 91.2262 83.2018 83.2688 89.9156]
Mean MSE: 87.4156 | RMSE: 9.3496 | Std: 3.4620

Running LightGBM_TFIDF...
Fold MSE: [86.1437 87.3735 77.58   77.7385 84.1276]
Mean MSE: 82.5927 | RMSE: 9.0881 | Std: 4.1596

Running XGBoost_TFIDF...
Fold MSE: [84.4007 86.7584 76.2881 76.939  85.1392]
Mean MSE: 81.9051 | RMSE: 9.0501 | Std: 4.3922

Running CatBoost_TFIDF...
Fold MSE: [82.5119 84.2474 73.902  75.3389 82.8236]
Mean MSE: 79.7647 | RMSE: 8.9311 | Std: 4.2651


,model,mean_mse,rmse,std_mse
4,CatBoost_TFIDF,79.764738,8.931111,4.265148
3,XGBoost_TFIDF,81.905082,9.050143,4.392153
2,LightGBM_TFIDF,82.592669,9.088051,4.159616
0,Ridge_TFIDF,87.361972,9.346763,3.445143
1,ElasticNet_TFIDF,87.415624,9.349632,3.461968


## 6. Choose best model and create submission

Pick the best model by CV MSE, then fit on full train and predict test.

Predictions are clipped to `[0, 100]` because the target is described as a 0–100 career success score.

In [17]:
best_model_name = results_df.iloc[0]["model"]
best_model = models[best_model_name]

print("Best model:", best_model_name)

best_model.fit(X, y)
test_predictions = best_model.predict(X_test)
test_predictions = np.clip(test_predictions, 0, 100)

submission = pd.DataFrame({
    ID_COL: test[ID_COL],
    TARGET: test_predictions
})

display(submission.head())
display(submission[TARGET].describe())

os.makedirs("../submissions", exist_ok=True)
submission_path = f"../submissions/submission_02_{best_model_name.lower()}.csv"
submission.to_csv(submission_path, index=False)

print("Saved submission to:", submission_path)

Best model: CatBoost_TFIDF


,student_id,career_success_score
0,STU_010001,58.012466
1,STU_010002,75.116198
2,STU_010003,73.552972
3,STU_010004,98.558365
4,STU_010005,79.005400


count    10000.000000
mean        76.134201
std         12.553066
min         34.034546
25%         67.330310
50%         75.922131
75%         85.365100
max        100.000000
Name: career_success_score, dtype: float64

Saved submission to: ../submissions/submission_02_catboost_tfidf.csv


## 7. Next steps

After running this notebook:

1. Compare the CV table with the Kaggle public score.
2. Do not overfit to the public leaderboard.
3. If CV and Kaggle disagree strongly, investigate validation mismatch.
4. Then move to `03_advanced_ensemble.ipynb` with out-of-fold predictions and careful blending.

Possible next improvements:

- role-specific features refined by target role,
- target distribution analysis by department / role / university tier,
- hyperparameter tuning for the best 2 models,
- out-of-fold ensemble instead of simple best-model submission,
- residual analysis to understand where the model fails.